In [3]:
import os
import re
import getpass
import requests
import openai
from openai import OpenAI
import jsonschema
import os
import json
from typing import List, Optional, Dict, Any, Tuple, Iterable
import math

import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from dataclasses import dataclass
from docx.document import Document as _Document
from docx.text.paragraph import Paragraph
from docx.table import Table
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl
from datetime import datetime


In [2]:
load_dotenv()

# ==== НАСТРОЙКИ ====
OPENAI_API_KEY = getpass.getpass("Введи ваш VseGPT ключ API:")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://api.vsegpt.ru/v1"
OPENAI_MODEL = "openai/gpt-4o-mini"

# Пути к файлам
CSV_CRITERIA_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\Критерии.csv"  # файл во вложении
DOCX_SPEC_PATH = r"C:\Users\troyd\OneDrive\Desktop\Стажировка\1_Приложение №1_ТЗ_сейсмика.docx"
OUTPUT_REPORT_DOCX = os.path.join(
    os.getcwd(),
    f"Analysis_Report_SGR_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.docx"
)

if not OPENAI_API_KEY or OPENAI_API_KEY == "YOUR_OPENAI_API_KEY_HERE":
    raise ValueError("Не задан OPENAI_API_KEY. Установите переменную окружения или впишите ключ.")

print("✅ Конфигурация загружена")

✅ Конфигурация загружена


In [9]:
class GeneratedCriterionItem(BaseModel):
    """
    ЕДИНЫЙ формат для "новых" критериев, сгенерированных LLM на этапе 1.
    Поле называется generated_criteria (канон), чтобы совпадать с промптом и JSON-schema.
    """
    name: str = Field(..., max_length=450, description="Название критерия (кратко и проверяемо по ТЗ).")
    description: str = Field(..., max_length=1500, description="Описание критерия (что именно проверять в ТЗ).")
    importance: int = Field(
        ...,
        ge=4,
        le=5,
        description="Важность 4–5 (генерируем только критичные критерии)."
    )

    class Config:
        extra = "forbid"


class GeneratedCriteria(BaseModel):
    """
    Результат ЭТАПА 1: отобранные ID из справочника + новые критерии, найденные в ТЗ.
    ВАЖНО: generated_criteria — единый канонический ключ.
    """
    selected_ids: List[int] = Field(
        default_factory=list,
        description="ID критериев из справочника, которые строго релевантны ТЗ."
    )
    generated_criteria: List[GeneratedCriterionItem] = Field(
        default_factory=list,
        max_length=5,
        description="Список новых критериев (макс. 5), строго вытекающих из текста ТЗ."
    )

    class Config:
        extra = "forbid"

class EvidenceQuote(BaseModel):
    """
    Одна дословная цитата из ТЗ + откуда она взята.
    chunk_id нужен, чтобы можно было отследить источник в переданном контексте.
    """
    chunk_id: int = Field(..., ge=1, description="Номер чанка (CHUNK N) из переданного текста ТЗ.")
    quote: str = Field(..., min_length=10, max_length=900, description="Дословная цитата из ТЗ.")
    
class CriterionReasoning(BaseModel):
    """Пошаговое рассуждение для одного критерия (Schema-Guided Reasoning)."""

    criterion_id: int = Field(..., description="Порядковый номер критерия")
    criterion_name: str = Field(..., description="Название критерия")
    criterion_description: str = Field(..., description="Описание из CSV")

    importance_level: int = Field(..., description="Определи уровень важности этого критерия для данного ТЗ")

    criterion_understanding: str = Field(..., description="Объясни, ЧТО проверяет этот критерий (1–2 предложения).")
    relevant_sections: str = Field(..., description="Какие разделы/приложения ТЗ релевантны для этого критерия?")
    reasoning_steps: str = Field(..., description="Пошагово объясни: Критерий требует X → В документе найдено Y → Вывод Z.")

    status: str = Field(
        ...,
        description="Статус выполнения критерия.",
        json_schema_extra={"enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]}
    )

    # ВАЖНО: теперь допускаем пустой список, если в предоставленных чанках нет дословных подтверждений.
    evidence_quotes: List[EvidenceQuote] = Field(
        default_factory=list,
        description="1–4 дословные цитаты из ТЗ (verbatim) с указанием chunk_id. "
                    "Если в предоставленных чанках нет подтверждений — передай пустой список."
    )

    recommendation: Optional[str] = Field(
        default=None,
        description="Рекомендация по улучшению. Если нет — передать null."
    )


class SpecificationAnalysisWithReasoning(BaseModel):
    """Анализ ТЗ с явным SGR для каждого критерия."""
    
    reasoning_schema_used: bool = Field(
        ..., # Убрали default=True, модель должна сама решить или вы жестко задаете это в промпте
        description="Флаг: используется ли Schema-Guided Reasoning."
    )

    overall_summary: str = Field(
        ...,
        description="Общая оценка качества ТЗ (1-2 абзаца)."
    )

    criteria_analysis: List[CriterionReasoning] = Field(
        ...,
        description="Список проверок ТОЛЬКО по критериям, отобранным как релевантные для данного ТЗ."
    )

    additional_criteria: List[CriterionReasoning] = Field(
        default=[],  # Может быть пустым списком, если модель не нашла дополнительных
        description="Дополнительные критерии, выявленные моделью при анализе ТЗ (проблемы, не покрытые базовым списком)."
    )
    
    # Лучше временно убрать Dict[str, Any] или заменить на конкретную модель, 
    # так как 'Any' плохо работает со строгим режимом.
    # Если метрики не критичны, можно закомментировать поле metrics:
    # metrics: Dict[str, str] = Field(..., description="Метрики анализа (ключ-значение).")


C:\Users\troyd\AppData\Local\Temp\ipykernel_15812\3598001731.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class GeneratedCriterionItem(BaseModel):
C:\Users\troyd\AppData\Local\Temp\ipykernel_15812\3598001731.py:19: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class GeneratedCriteria(BaseModel):


In [10]:
def load_criteria_from_csv(csv_path: str) -> List[Dict[str, Any]]:
    """
    Читает критерии из CSV с разделителем ';' (точка с запятой).
    Ожидаемые колонки: Название, Описание, Важность
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV с критериями не найден: {csv_path}")
    
    # Важно: использовать sep=';' и encoding зависит от системы
    df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    
    print(f"Загруженные колонки: {df.columns.tolist()}")
    print(f"Первая строка: {df.iloc[0].to_dict()}")
    
    # Проверка обязательных колонок
    required_cols = ["Название", "Описание", "Важность"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"В CSV отсутствует обязательная колонка '{col}'. Колонки: {df.columns.tolist()}")
    
    criteria = []
    for idx, row in df.iterrows():
        criteria.append({
            "id": idx + 1,  # Порядковый номер
            "name": str(row["Название"]).strip(),
            "description": str(row["Описание"]).strip(),
            "importance": int(row["Важность"]),
        })
    
    print(f"✅ Загружено критериев: {len(criteria)}")
    return criteria


def _iter_block_items_in_order(doc: _Document) -> Iterable[Tuple[str, object]]:
    """
    Итерирует элементы DOCX в исходном порядке (paragraphs + tables).
    Возвращает ('p', Paragraph) или ('tbl', Table).
    """
    parent = doc.element.body
    for child in parent.iterchildren():
        if isinstance(child, CT_P):
            yield ("p", Paragraph(child, doc))
        elif isinstance(child, CT_Tbl):
            yield ("tbl", Table(child, doc))


def _cell_text(cell) -> str:
    # Берём и paragraphs, и возможные вложенные таблицы (на всякий случай)
    parts = []
    for p in cell.paragraphs:
        t = (p.text or "").strip()
        if t:
            parts.append(t)
    # Вложенные таблицы
    for t in getattr(cell, "tables", []) or []:
        parts.append(_table_text(t))
    return "\n".join(parts).strip()


def _table_text(table: Table) -> str:
    rows_out = []
    for r_i, row in enumerate(table.rows, start=1):
        cells = []
        for c_i, cell in enumerate(row.cells, start=1):
            txt = _cell_text(cell)
            txt = re.sub(r"\s+", " ", txt).strip()
            cells.append(txt)
        # Убираем полностью пустые строки таблицы
        if any(c for c in cells if c):
            rows_out.append(" | ".join(cells))
    return "\n".join(rows_out).strip()


def extract_text_from_docx(docx_path: str) -> str:
    """
    Извлекает текст из DOCX:
    - параграфы
    - таблицы (включая текст из ячеек)
    ВАЖНО: сохраняет порядок появления в документе.
    """
    if not os.path.exists(docx_path):
        raise FileNotFoundError(f"DOCX файл не найден: {docx_path}")

    doc = Document(docx_path)

    out_lines = []
    p_idx = 0
    t_idx = 0

    for kind, obj in _iter_block_items_in_order(doc):
        if kind == "p":
            txt = (obj.text or "").strip()
            if txt:
                p_idx += 1
                out_lines.append(f"[P{p_idx}] {txt}")
        else:
            table_txt = _table_text(obj)
            if table_txt:
                t_idx += 1
                out_lines.append(f"[TABLE {t_idx}]\n{table_txt}")

    full_text = "\n\n".join(out_lines).strip()
    print(f"✅ Текст ТЗ извлечён (параграфов={p_idx}, таблиц={t_idx}, символов={len(full_text)})")
    return full_text


def chunk_text(
    text: str,
    max_chars: int = 12000,
    overlap_chars: int = 800,
) -> List[str]:
    """
    Чанкинг по смысловым блокам (разделитель: двойной перенос строки),
    с небольшим overlap по символам, чтобы не терять контекст на границах.
    """
    text = (text or "").strip()
    if not text:
        return []

    blocks = [b.strip() for b in text.split("\n\n") if b.strip()]
    chunks: List[str] = []
    buf = ""

    def flush():
        nonlocal buf
        if buf.strip():
            chunks.append(buf.strip())
        buf = ""

    for b in blocks:
        candidate = (buf + "\n\n" + b).strip() if buf else b
        if len(candidate) <= max_chars:
            buf = candidate
            continue

        # если буфер уже есть — сбрасываем
        if buf:
            flush()
            # overlap: хвост предыдущего чанка
            tail = chunks[-1][-overlap_chars:].strip() if chunks and overlap_chars > 0 else ""
            buf = (tail + "\n\n" + b).strip() if tail else b
            # если всё равно слишком большой одиночный блок — режем грубо
            if len(buf) > max_chars:
                while len(buf) > max_chars:
                    part = buf[:max_chars].strip()
                    chunks.append(part)
                    buf = buf[max_chars - overlap_chars :].strip() if overlap_chars > 0 else buf[max_chars:].strip()
                buf = buf.strip()
        else:
            # одиночный блок больше max_chars — режем грубо
            big = b
            while len(big) > max_chars:
                chunks.append(big[:max_chars].strip())
                big = big[max_chars - overlap_chars :].strip() if overlap_chars > 0 else big[max_chars:].strip()
            buf = big

    flush()
    return chunks


def prepare_spec_text(
    docx_path: str,
    *,
    max_chars_per_chunk: int = 12000,
    overlap_chars: int = 800,
) -> Dict[str, Any]:
    """
    ЕДИНАЯ подготовка ТЗ:
    1) извлечение текста (параграфы + таблицы)
    2) нормализация пробелов/переносов
    3) чанкинг
    Возвращает dict: { "full_text": ..., "chunks": [...], "meta": {...} }
    """
    raw = extract_text_from_docx(docx_path)

    # Нормализация: не убиваем маркеры [P#]/[TABLE #], но чистим лишние пробелы
    lines = []
    for line in raw.splitlines():
        if line.strip():
            lines.append(re.sub(r"[ \t]+", " ", line).rstrip())
    normalized = "\n".join(lines).strip()

    chunks = chunk_text(normalized, max_chars=max_chars_per_chunk, overlap_chars=overlap_chars)

    # Добавим заголовки чанков (удобно для цитирования и пост-валидации)
    labeled_chunks = []
    for i, ch in enumerate(chunks, start=1):
        labeled_chunks.append(f"=== CHUNK {i}/{len(chunks)} ===\n{ch}")

    return {
        "full_text": normalized,
        "chunks": labeled_chunks,
        "meta": {
            "source_docx": os.path.basename(docx_path),
            "chars_total": len(normalized),
            "chunks_count": len(labeled_chunks),
            "max_chars_per_chunk": max_chars_per_chunk,
            "overlap_chars": overlap_chars,
        },
    }

_WORD_RE = re.compile(r"[A-Za-zА-Яа-я0-9_]+", re.UNICODE)

def _tokenize(text: str) -> List[str]:
    return [t.lower() for t in _WORD_RE.findall(text or "")]

def _strip_chunk_header(chunk: str) -> str:
    """
    Убираем сервисный заголовок вида '=== CHUNK i/n ===' чтобы проверки цитат работали по реальному контенту.
    """
    if not chunk:
        return ""
    lines = chunk.splitlines()
    if lines and lines[0].strip().startswith("=== CHUNK"):
        return "\n".join(lines[1:]).strip()
    return chunk.strip()

def build_chunk_map(labeled_chunks: List[str]) -> Dict[int, str]:
    """
    Возвращает {chunk_id: chunk_text_without_header}
    chunk_id соответствует номеру в '=== CHUNK i/n ==='
    """
    out: Dict[int, str] = {}
    for i, ch in enumerate(labeled_chunks, start=1):
        out[i] = _strip_chunk_header(ch)
    return out

@dataclass
class BM25Index:
    chunks: List[Tuple[int, str]]          # (chunk_id, chunk_text)
    doc_freq: Dict[str, int]
    avgdl: float
    doc_lens: List[int]
    tf_maps: List[Dict[str, int]]

def build_bm25_index(chunk_map: Dict[int, str]) -> BM25Index:
    chunks = list(chunk_map.items())  # [(id, text)]
    tf_maps: List[Dict[str, int]] = []
    doc_freq: Dict[str, int] = {}
    doc_lens: List[int] = []
    total_len = 0

    for _, txt in chunks:
        toks = _tokenize(txt)
        total_len += len(toks)
        doc_lens.append(len(toks))
        tf: Dict[str, int] = {}
        for t in toks:
            tf[t] = tf.get(t, 0) + 1
        tf_maps.append(tf)
        for t in tf.keys():
            doc_freq[t] = doc_freq.get(t, 0) + 1

    avgdl = (total_len / len(chunks)) if chunks else 0.0
    return BM25Index(chunks=chunks, doc_freq=doc_freq, avgdl=avgdl, doc_lens=doc_lens, tf_maps=tf_maps)

def bm25_top_k(index: BM25Index, query: str, k: int = 4, *, k1: float = 1.5, b: float = 0.75) -> List[int]:
    """
    Возвращает top-k chunk_id по BM25.
    """
    q_tokens = _tokenize(query)
    if not q_tokens or not index.chunks:
        return [cid for cid, _ in index.chunks[:k]]

    N = len(index.chunks)
    scores: List[Tuple[float, int]] = []

    for doc_i, (chunk_id, _) in enumerate(index.chunks):
        tf = index.tf_maps[doc_i]
        dl = index.doc_lens[doc_i] or 1
        score = 0.0
        for t in q_tokens:
            if t not in tf:
                continue
            df = index.doc_freq.get(t, 0)
            # IDF (BM25+ стандартная форма)
            idf = math.log(1.0 + (N - df + 0.5) / (df + 0.5))
            freq = tf[t]
            denom = freq + k1 * (1.0 - b + b * (dl / (index.avgdl or 1.0)))
            score += idf * (freq * (k1 + 1.0)) / (denom or 1.0)
        scores.append((score, chunk_id))

    scores.sort(reverse=True, key=lambda x: x[0])
    top = [cid for _, cid in scores[:k]]
    # если всё нулевое — вернем первые k
    if all(s == 0.0 for s, _ in scores[:k]):
        return [cid for cid, _ in index.chunks[:k]]
    return top

def _norm_for_match(s: str) -> str:
    """
    Нормализация для поиска 'verbatim' с устойчивостью к лишним пробелам/переносам.
    ВАЖНО: это не "семантика", только whitespace-normalization.
    """
    s = (s or "").replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    return s.strip()

def quote_is_verbatim_in_chunk(quote: str, chunk_text: str) -> bool:
    """
    True если quote (после whitespace-нормализации) является подстрокой chunk_text (после нормализации).
    """
    q = _norm_for_match(quote)
    c = _norm_for_match(chunk_text)
    if not q or not c:
        return False
    return q in c

def validate_evidence_quotes(item: Dict[str, Any], chunk_map: Dict[int, str]) -> List[str]:
    """
    Возвращает список ошибок grounding по evidence_quotes.
    """
    errs: List[str] = []
    quotes = item.get("evidence_quotes", [])
    if quotes is None:
        errs.append("evidence_quotes is null (must be [] or list)")
        return errs
    if not isinstance(quotes, list):
        errs.append("evidence_quotes is not a list")
        return errs

    for qi, q in enumerate(quotes, start=1):
        if not isinstance(q, dict):
            errs.append(f"evidence_quotes[{qi}] is not an object")
            continue
        cid = q.get("chunk_id")
        qt = q.get("quote")
        if not isinstance(cid, int) or cid < 1:
            errs.append(f"evidence_quotes[{qi}].chunk_id invalid: {cid}")
            continue
        if cid not in chunk_map:
            errs.append(f"evidence_quotes[{qi}].chunk_id out of range: {cid}")
            continue
        if not isinstance(qt, str) or not qt.strip():
            errs.append(f"evidence_quotes[{qi}].quote empty")
            continue
        if not quote_is_verbatim_in_chunk(qt, chunk_map[cid]):
            errs.append(f"evidence_quotes[{qi}] quote not found verbatim in chunk {cid}")
    return errs

def format_chunks_for_prompt(chunk_ids: List[int], chunk_map: Dict[int, str], *, max_chars_total: int = 18000) -> str:
    """
    Собирает контекст из выбранных чанков с жёстким лимитом символов.
    """
    out = []
    used = 0
    for cid in chunk_ids:
        txt = chunk_map.get(cid, "")
        block = f"CHUNK {cid}:\n{txt}\n"
        if used + len(block) > max_chars_total:
            break
        out.append(block)
        used += len(block)
    return "\n".join(out).strip()

In [11]:
def create_criteria_selection_prompt(spec_text: str, all_criteria: List[Dict[str, Any]]) -> str:
    """
    ЭТАП 1: Анализ ТЗ и формирование списка проверок.
    """
    criteria_list_str = "\n".join([
        f"{c['id']}. {c['name']} (Важность: {c['importance']})\n {c['description']}"
        for c in all_criteria
    ])
    
    return f"""
Ты — методолог по системному анализу. Твоя задача — изучить ТЗ и составить достаточный список критериев для проверки.

СТРОГИЕ ПРАВИЛА:
1. Используй ТОЛЬКО информацию из текста ТЗ. Запрещено опираться на внешние знания, стандарты или практики, не упомянутые в ТЗ.
2. Если в ТЗ нет информации по какому-либо аспекту — НЕ генерируй критерий для его проверки.
3. Максимум 5 новых критерия. Генерируй их только если они:
   - Явно вытекают из текста ТЗ,
   - Критически важны (важность 4–5),
   - Проверяемы по тексту ТЗ без домыслов.

Алгоритм работы:
1. Проанализируй ТЗ: определи тип системы и предметную область.
2. Выбери из Справочника ID тех критериев, которые применимы к этому ТЗ.
3. Если есть критические аспекты, не покрытые Справочником и явно присутствующие в ТЗ — добавь не более 5 новых критериев.

ЗАПРЕЩЕНО:
- Генерировать критерии для аспектов, не упомянутых в ТЗ,
- Предполагать наличие модулей, ролей, интеграций, не описанных в ТЗ.

СПРАВОЧНИК КРИТЕРИЕВ:
{criteria_list_str}

ТЕХНИЧЕСКОЕ ЗАДАНИЕ:
{spec_text[:8000]}

Верни JSON с полями:
- selected_ids: массив ID из справочника,
- generated_criteria: массив новых критериев (максимум 5, можно пустой массив).
"""


def create_analysis_prompt(spec_context: str, final_criteria: List[Dict[str, Any]]) -> str:
    """
    ЭТАП 2: Детальный анализ по утвержденному списку.
    Контекст содержит ТОЛЬКО выбранные чанки (retrieve-then-read).
    """
    criteria_list = "\n".join([
        f"ID {c['id']}: {c['name']}\nОписание: {c['description']}"
        for c in final_criteria
    ])

    return f"""
Ты — системный аудитор технической документации. Проведи аудит ТЗ по заданному списку критериев.

КРИТИЧЕСКИ ВАЖНЫЕ ПРАВИЛА (GROUNDING):
1) Используй ТОЛЬКО текст в предоставленных CHUNK. Внешние знания/стандарты/догадки запрещены.
2) evidence_quotes:
   - Добавляй цитаты ТОЛЬКО если они существуют ДОСЛОВНО (verbatim) в тексте CHUNK.
   - Каждая цитата должна быть точной подстрокой из указанного chunk_id.
   - Если в предоставленных чанках НЕТ дословного подтверждения — evidence_quotes = [].
3) НЕЛЬЗЯ выдумывать цитаты или перефразировать как “цитату”.
4) НЕЛЬЗЯ писать “обычно/вероятно/можно предположить”.

Если по критерию нет подтверждений в предоставленных чанках:
- status = "❌ Не выполнено"
- evidence_quotes = []
- reasoning_steps: явно опиши, что искал в данных чанках и что подтверждения не найдено.

КОНТЕКСТ ТЗ (ТОЛЬКО ЭТИ ФРАГМЕНТЫ ДОСТУПНЫ):
{spec_context}

КРИТЕРИИ ДЛЯ ПРОВЕРКИ:
{criteria_list}

Верни JSON строго по схеме.
"""



In [14]:
class SpecAnalyzerWithSGR:
    """Анализатор ТЗ с двухэтапным процессом: Генерация критериев -> SGR Анализ."""

    def __init__(self, api_key: str, model: str = "openai/gpt-4o-mini"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.vsegpt.ru/v1"
        )
        self.model = model
        print(f"✅ Инициализирован анализатор VseGPT ({model})")

    def select_and_generate_criteria(
        self,
        spec_text: str,
        all_criteria: List[Dict[str, Any]],
        *,
        spec_chunks: Optional[List[str]] = None,
        max_chunks_for_stage1: int = 8,
    ) -> List[Dict[str, Any]]:
        """
        ЭТАП 1: Выбирает существующие и генерирует новые критерии.
        ВАЖНО: используем чанкинг, чтобы не переполнять контекст.
        """
        # Если чанки не передали — нарежем сами (без DOCX, просто по тексту)
        chunks = spec_chunks or chunk_text(spec_text, max_chars=12000, overlap_chars=800)
        if not chunks:
            raise ValueError("spec_text пустой — нечего анализировать")

        chunks_to_use = chunks[:max_chunks_for_stage1]

        generation_schema = {
            "name": "CriteriaSelection",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "selected_ids": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "description": "Список ID критериев из справочника, которые строго применимы к этому ТЗ."
                    },
                    "generated_criteria": {
                        "type": "array",
                        "maxItems": 5,
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string", "maxLength": 450},
                                "description": {"type": "string", "maxLength": 1500},
                                "importance": {
                                    "type": "integer",
                                    "minimum": 4,
                                    "maximum": 5,
                                    "description": "Важность 4–5 (генерируем только критичные)."
                                }
                            },
                            "required": ["name", "description", "importance"],
                            "additionalProperties": False
                        },
                        "description": "Список НОВЫХ критериев (макс. 5) по этому фрагменту ТЗ."
                    }
                },
                "required": ["selected_ids", "generated_criteria"],
                "additionalProperties": False
            }
        }

        print(f"🕵️ ЭТАП 1 (chunked): чанков всего={len(chunks)}, используем={len(chunks_to_use)}")

        selected_ids_union: set[int] = set()
        generated_pool: List[Dict[str, Any]] = []
        generated_seen_key: set[str] = set()

        for i, ch in enumerate(chunks_to_use, start=1):
            prompt = create_criteria_selection_prompt(ch, all_criteria)

            try:
                response = self.client.chat.completions.create(
                    model=self.model,
                    temperature=0.2,
                    top_p=0.85,
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_schema", "json_schema": generation_schema}
                )
                data = json.loads(response.choices[0].message.content)

                for _id in data.get("selected_ids", []) or []:
                    if isinstance(_id, int):
                        selected_ids_union.add(_id)

                for nc in data.get("generated_criteria", []) or []:
                    name = (nc.get("name") or "").strip()
                    desc = (nc.get("description") or "").strip()
                    imp = nc.get("importance")
                    key = (name.lower() + "||" + desc.lower())[:8000]

                    if not name or not desc:
                        continue
                    if key in generated_seen_key:
                        continue
                    generated_seen_key.add(key)

                    generated_pool.append({
                        "name": name,
                        "description": desc,
                        "importance": int(imp) if imp is not None else 4
                    })

                print(f"   ✅ chunk {i}/{len(chunks_to_use)}: selected_ids={len(data.get('selected_ids', []) or [])}, new={len(data.get('generated_criteria', []) or [])}")
            except Exception as e:
                print(f"❌ Ошибка на этапе 1 (chunk {i}): {e}")
                raise

        # Сборка финального списка
        final_list: List[Dict[str, Any]] = []

        # 1) выбранные из CSV
        for crit in all_criteria:
            if crit["id"] in selected_ids_union:
                final_list.append(crit)

        # 2) сгенерированные — присваиваем новые ID
        max_id = max([c["id"] for c in all_criteria], default=0)
        current_id = max_id + 1

        # ограничим общий пул новых критериев (на случай суммирования по чанкам)
        generated_pool = generated_pool[:5]

        for new_crit in generated_pool:
            final_list.append({
                "id": current_id,
                "name": f"[AI] {new_crit['name']}",
                "description": new_crit["description"],
                "importance": int(new_crit["importance"]),
            })
            current_id += 1

        print(f"   📌 Итого selected_ids (union): {len(selected_ids_union)}")
        print(f"   ✨ Итого new criteria (dedup): {len(generated_pool)}")
        print(f"   📋 Итого к проверке: {len(final_list)}")

        return final_list

    def _analyze_single_criterion(
        self,
        *,
        criterion: Dict[str, Any],
        spec_context: str,
    ) -> Dict[str, Any]:
        """
        Один вызов модели на один критерий (строго по json_schema).
        Возвращает dict (один CriterionReasoning).
        """
        # Мы прокидываем в create_analysis_prompt "final_criteria" как список из 1 критерия.
        prompt = create_analysis_prompt(spec_context, [criterion])

        json_schema = {
            "name": "CriterionReasoning",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "criterion_id": {"type": "integer"},
                    "criterion_name": {"type": "string"},
                    "criterion_description": {"type": "string"},
                    "importance_level": {"type": "integer", "minimum": 1, "maximum": 5},
                    "criterion_understanding": {"type": "string", "maxLength": 1400},
                    "relevant_sections": {"type": "string", "maxLength": 1300},
                    "reasoning_steps": {"type": "string", "maxLength": 1800},
                    "status": {"type": "string", "enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]},
                    "evidence_quotes": {
                        "type": "array",
                        "minItems": 0,
                        "maxItems": 4,
                        "items": {
                            "type": "object",
                            "properties": {
                                "chunk_id": {"type": "integer", "minimum": 1},
                                "quote": {"type": "string", "minLength": 10, "maxLength": 900}
                            },
                            "required": ["chunk_id", "quote"],
                            "additionalProperties": False
                        }
                    },
                    "recommendation": {"type": ["string", "null"]},
                },
                "required": [
                    "criterion_id", "criterion_name", "criterion_description",
                    "importance_level", "criterion_understanding", "relevant_sections",
                    "reasoning_steps", "status", "evidence_quotes", "recommendation"
                ],
                "additionalProperties": False
            }
        }

        resp = self.client.chat.completions.create(
            model=self.model,
            temperature=0.1,
            top_p=0.8,
            max_tokens=2500,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema", "json_schema": json_schema}
        )
        return json.loads(resp.choices[0].message.content)


    def _repair_single_criterion_grounding(
        self,
        *,
        broken_item: Dict[str, Any],
        errors: List[str],
        criterion: Dict[str, Any],
        spec_context: str,
    ) -> Dict[str, Any]:
        """
        Repair loop: заставляем модель заменить evidence_quotes так, чтобы каждая цитата
        существовала verbatim в указанных чанках. Если нет — evidence_quotes=[] и статус ❌.
        """
        repair_prompt = f"""
    Ты сделал вывод по критерию, но пост-валидация grounding нашла ошибки.

    КРИТЕРИЙ:
    ID {criterion['id']}: {criterion['name']}
    Описание: {criterion['description']}

    КОНТЕКСТ (ЕДИНСТВЕННЫЙ ИСТОЧНИК):
    {spec_context}

    ТВОЙ ТЕКУЩИЙ JSON (может содержать ошибки):
    {json.dumps(broken_item, ensure_ascii=False)}

    ОШИБКИ GROUNDING (исправь ВСЕ):
    - """ + "\n- ".join(errors) + """

    ПРАВИЛА ИСПРАВЛЕНИЯ:
    1) Любая evidence_quotes[i].quote должна быть точной подстрокой (verbatim) текста соответствующего CHUNK.
    2) НЕЛЬЗЯ выдумывать или перефразировать цитаты.
    3) Если в этих чанках нет дословных подтверждений — evidence_quotes = [] и status="❌ Не выполнено".
    4) Верни JSON строго по схеме CriterionReasoning.
    """

        json_schema = {
            "name": "CriterionReasoning",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "criterion_id": {"type": "integer"},
                    "criterion_name": {"type": "string"},
                    "criterion_description": {"type": "string"},
                    "importance_level": {"type": "integer", "minimum": 1, "maximum": 5},
                    "criterion_understanding": {"type": "string", "maxLength": 1400},
                    "relevant_sections": {"type": "string", "maxLength": 1300},
                    "reasoning_steps": {"type": "string", "maxLength": 1800},
                    "status": {"type": "string", "enum": ["✅ Выполнено", "⚠️ Частично", "❌ Не выполнено"]},
                    "evidence_quotes": {
                        "type": "array",
                        "minItems": 0,
                        "maxItems": 4,
                        "items": {
                            "type": "object",
                            "properties": {
                                "chunk_id": {"type": "integer", "minimum": 1},
                                "quote": {"type": "string", "minLength": 10, "maxLength": 900}
                            },
                            "required": ["chunk_id", "quote"],
                            "additionalProperties": False
                        }
                    },
                    "recommendation": {"type": ["string", "null"]},
                },
                "required": [
                    "criterion_id", "criterion_name", "criterion_description",
                    "importance_level", "criterion_understanding", "relevant_sections",
                    "reasoning_steps", "status", "evidence_quotes", "recommendation"
                ],
                "additionalProperties": False
            }
        }

        resp = self.client.chat.completions.create(
            model=self.model,
            temperature=0.0,  # максимально детерминированно на ремонте
            top_p=1.0,
            max_tokens=2500,
            messages=[{"role": "user", "content": repair_prompt}],
            response_format={"type": "json_schema", "json_schema": json_schema}
        )
        return json.loads(resp.choices[0].message.content)

    def analyze_with_sgr(
        self,
        spec_text: str,
        criteria: List[Dict[str, Any]],
        *,
        spec_chunks: Optional[List[str]] = None,
        top_k_chunks_per_criterion: int = 4,
        max_chars_per_criterion_context: int = 18000,
        max_repairs_per_criterion: int = 2,
    ) -> SpecificationAnalysisWithReasoning:
        """
        ЭТАП 2: SGR-анализ (production-grade):
        - retrieve-then-read: на каждый критерий берём top-K чанков
        - строгий grounding: evidence_quotes должны быть verbatim подстрокой в chunk_id
        - post-validation + repair loop
        - контроль контекста: лимит символов на критерий
        """
        chunks = spec_chunks or chunk_text(spec_text, max_chars=12000, overlap_chars=800)
        if not chunks:
            raise ValueError("spec_text пустой — нечего анализировать")

        chunk_map = build_chunk_map(chunks)            # {chunk_id: text}
        bm25 = build_bm25_index(chunk_map)

        print(f"🔬 ЭТАП 2 (retrieve-then-read): критериев={len(criteria)}, чанков={len(chunks)}, topK={top_k_chunks_per_criterion}")

        criteria_items: List[Dict[str, Any]] = []
        had_any_repairs = False

        for idx, c in enumerate(criteria, start=1):
            query = f"{c.get('name','')} {c.get('description','')}".strip()
            top_ids = bm25_top_k(bm25, query, k=top_k_chunks_per_criterion)
            spec_context = format_chunks_for_prompt(top_ids, chunk_map, max_chars_total=max_chars_per_criterion_context)

            # 1) основной анализ
            item = self._analyze_single_criterion(criterion=c, spec_context=spec_context)

            # 2) post-validation grounding
            errs = validate_evidence_quotes(item, chunk_map)

            # 3) repair loop
            repairs = 0
            while errs and repairs < max_repairs_per_criterion:
                had_any_repairs = True
                repairs += 1
                item = self._repair_single_criterion_grounding(
                    broken_item=item,
                    errors=errs,
                    criterion=c,
                    spec_context=spec_context
                )
                errs = validate_evidence_quotes(item, chunk_map)

            # 4) hard fallback: если после ремонтов всё ещё есть ошибки — вычищаем evidence_quotes
            if errs:
                print(f"⚠️ Grounding не починен для критерия #{idx} после {repairs} repair(ов): {errs}")
                item["evidence_quotes"] = []
                item["status"] = "❌ Не выполнено"
                item["reasoning_steps"] = (item.get("reasoning_steps", "").strip() + "\n\n"
                                        "POST-VALIDATION: доказательные цитаты были отклонены (не найдены verbatim в чанках).").strip()

            criteria_items.append(item)

        overall_summary = (
            "Отчет сформирован в режиме retrieve-then-read с пост-валидацией цитат. "
            + ("Были выполнены repair-циклы для части критериев." if had_any_repairs else "Repair-циклы не потребовались.")
        )

        data = {
            "reasoning_schema_used": True,
            "overall_summary": overall_summary[:1500],
            "criteria_analysis": criteria_items,
            "additional_criteria": [],
        }

        return SpecificationAnalysisWithReasoning(**data)
     

In [15]:
def _safe_text(value, max_len: int = 5000) -> str:
    """
    Безопасное преобразование текста перед записью в DOCX.
    Убирает None, лишние переводы строк и слишком длинные строки.
    """
    if value is None:
        return ""

    txt = str(value)

    # нормализуем переносы строк
    txt = txt.replace("\r\n", "\n").replace("\r", "\n")

    # убираем невидимые проблемные символы
    txt = txt.replace("\x00", "").strip()

    # ограничение длины (docx может падать на огромных строках)
    if len(txt) > max_len:
        txt = txt[:max_len] + " …[TRUNCATED]"

    return txt


def export_sgr_analysis_to_docx(
    analysis,
    source_docx_path: str,
    output_path: str
):
    """
    Стабильный экспорт отчёта SGR в DOCX.

    Защиты:
    - safe_text для всех полей
    - защита от None
    - устойчивость к неожиданным данным модели
    """

    print("📝 Экспорт отчёта в DOCX...")

    doc = Document()

    # --- Заголовок ---
    title = doc.add_heading("SGR Анализ технического задания", 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph(f"Источник: {_safe_text(source_docx_path)}")
    doc.add_paragraph(f"Дата генерации: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    doc.add_paragraph("")

    # --- Общий вывод ---
    doc.add_heading("Общий вывод", level=1)
    doc.add_paragraph(_safe_text(getattr(analysis, "overall_summary", "")))

    # --- Анализ критериев ---
    doc.add_heading("Анализ критериев", level=1)

    criteria_list = getattr(analysis, "criteria_analysis", []) or []

    for idx, crit in enumerate(criteria_list, start=1):
        try:
            name = _safe_text(getattr(crit, "criterion_name", f"Criterion {idx}"))
            importance = getattr(crit, "importance_level", "?")
            status = _safe_text(getattr(crit, "status", "—"))

            header = f"{idx}. {name} (важность: {importance}) — {status}"
            doc.add_heading(header, level=2)

            doc.add_paragraph("Описание:")
            doc.add_paragraph(_safe_text(getattr(crit, "criterion_description", "")))

            doc.add_paragraph("Понимание критерия:")
            doc.add_paragraph(_safe_text(getattr(crit, "criterion_understanding", "")))

            doc.add_paragraph("Релевантные разделы:")
            doc.add_paragraph(_safe_text(getattr(crit, "relevant_sections", "")))

            doc.add_paragraph("Логика анализа:")
            doc.add_paragraph(_safe_text(getattr(crit, "reasoning_steps", "")))

            doc.add_paragraph("Цитаты из ТЗ (доказательства):")

            doc.add_paragraph("Доказательства (цитаты из ТЗ):")
            ev_list = getattr(crit, "evidence_quotes", None) or []
            if not ev_list:
                doc.add_paragraph("— Нет дословных подтверждений в предоставленных чанках (evidence_quotes = []).")
            else:
                for q in ev_list:
                    try:
                        cid = getattr(q, "chunk_id", "?")
                        qt = _safe_text(getattr(q, "quote", ""))
                        doc.add_paragraph(f"[CHUNK {cid}] «{qt}»")
                    except Exception:
                        doc.add_paragraph("— (ошибка чтения evidence_quote)")

            recommendation = _safe_text(getattr(crit, "recommendation", ""))

            if recommendation:
                doc.add_paragraph("Рекомендация:")
                doc.add_paragraph(recommendation)

        except Exception as e:
            # Не даём экспорту падать из-за одного кривого критерия
            doc.add_paragraph(f"[Ошибка отображения критерия {idx}: {e}]")

    # --- Сохранение ---
    try:
        doc.save(output_path)
        print(f"✅ Отчёт сохранён: {output_path}")
    except Exception as e:
        print(f"❌ Ошибка сохранения DOCX: {e}")
        raise

In [16]:
print("=" * 60)
print("ДВУХЭТАПНЫЙ АНАЛИЗ ТЗ: GROUNDING + RETRIEVE-THEN-READ")
print("=" * 60)

# 1) Критерии
base_criteria = load_criteria_from_csv(CSV_CRITERIA_PATH)

# 2) Подготовка ТЗ
prepared = prepare_spec_text(
    DOCX_SPEC_PATH,
    max_chars_per_chunk=12000,
    overlap_chars=800
)
spec_text = prepared["full_text"]
spec_chunks = prepared["chunks"]
print("✅ prepare_spec_text meta:", prepared["meta"])

# 3) Анализатор
analyzer_sgr = SpecAnalyzerWithSGR(api_key=OPENAI_API_KEY, model=OPENAI_MODEL)

# 4) Stage1
final_criteria_pool = analyzer_sgr.select_and_generate_criteria(
    spec_text=spec_text,
    all_criteria=base_criteria,
    spec_chunks=spec_chunks,
    max_chunks_for_stage1=8
)

# 5) Stage2 (retrieve-then-read + grounding validation + repair)
analysis_result = analyzer_sgr.analyze_with_sgr(
    spec_text=spec_text,
    criteria=final_criteria_pool,
    spec_chunks=spec_chunks,
    top_k_chunks_per_criterion=4,
    max_chars_per_criterion_context=18000,
    max_repairs_per_criterion=2
)

print("✅ Анализ завершён")

# 6) Export
export_sgr_analysis_to_docx(
    analysis=analysis_result,
    source_docx_path=DOCX_SPEC_PATH,
    output_path=OUTPUT_REPORT_DOCX
)

print("Файл сохранён:", OUTPUT_REPORT_DOCX)
print("=" * 60)
print("✅ ГОТОВО! Отчет сформирован.")
print("=" * 60)

ДВУХЭТАПНЫЙ АНАЛИЗ ТЗ: GROUNDING + RETRIEVE-THEN-READ
Загруженные колонки: ['Название', 'Описание', 'Важность']
Первая строка: {'Название': 'Полнота функциональных требований', 'Описание': 'Оценка того, насколько подробно описано, что должна делать система, включая объекты, бизнес-логику, роли и отчётность.', 'Важность': 3}
✅ Загружено критериев: 26
✅ Текст ТЗ извлечён (параграфов=62, таблиц=1, символов=9812)
✅ prepare_spec_text meta: {'source_docx': '1_Приложение №1_ТЗ_сейсмика.docx', 'chars_total': 9750, 'chunks_count': 1, 'max_chars_per_chunk': 12000, 'overlap_chars': 800}
✅ Инициализирован анализатор VseGPT (openai/gpt-4o-mini)
🕵️ ЭТАП 1 (chunked): чанков всего=1, используем=1
   ✅ chunk 1/1: selected_ids=5, new=5
   📌 Итого selected_ids (union): 5
   ✨ Итого new criteria (dedup): 5
   📋 Итого к проверке: 10
🔬 ЭТАП 2 (retrieve-then-read): критериев=10, чанков=1, topK=4
⚠️ Grounding не починен для критерия #10 после 2 repair(ов): ['evidence_quotes[1].chunk_id out of range: 18', 'evi